In [ ]:
from pyspark.sql.functions import col, trim
from delta.tables import DeltaTable
import sys

sys.path.append("/Workspace/Users/sivana9908_gmail.com#ext#@sivana9908gmail.onmicrosoft.com/Uber-Eats-End-to-End-_Azure-Data-Engineering-Project")
from src.common.spark_utils import standardize_columns

In [ ]:
# Read the Bronze source.
df_zones_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("abfss://bronze@ubereaststorage.dfs.core.windows.net/csv/zones/")
)

source_rows = df_zones_raw.count()
print(f"zones.source_rows={source_rows}")
display(df_zones_raw)
df_zones_raw.printSchema()

In [ ]:
# Standardize columns and capture source quality metrics before publishing.
df_zones = standardize_columns(df_zones_raw)

null_zone_ids = df_zones.filter(
    col("zone_id").isNull() | (trim(col("zone_id").cast("string")) == "")
).count()

duplicate_zone_keys = (
    df_zones
    .filter(col("zone_id").isNotNull())
    .groupBy("zone_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

missing_city_rows = df_zones.filter(
    col("city").isNull() | (trim(col("city")) == "")
).count()
missing_zone_name_rows = df_zones.filter(
    col("zone_name").isNull() | (trim(col("zone_name")) == "")
).count()

latitude_value = col("latitude").cast("double")
longitude_value = col("longitude").cast("double")

invalid_latitude_rows = df_zones.filter(
    latitude_value.isNull()
    | (latitude_value < -90)
    | (latitude_value > 90)
).count()
invalid_longitude_rows = df_zones.filter(
    longitude_value.isNull()
    | (longitude_value < -180)
    | (longitude_value > 180)
).count()

print(f"zones.quality.null_or_blank_zone_id_rows={null_zone_ids}")
print(f"zones.quality.duplicate_zone_id_keys={duplicate_zone_keys}")
print(f"zones.quality.missing_city_rows={missing_city_rows}")
print(f"zones.quality.missing_zone_name_rows={missing_zone_name_rows}")
print(f"zones.quality.invalid_latitude_rows={invalid_latitude_rows}")
print(f"zones.quality.invalid_longitude_rows={invalid_longitude_rows}")

# Fail before writing if merge keys are unsafe or coordinates are invalid.
# This avoids silently publishing ambiguous zone IDs or unusable map locations.
if null_zone_ids:
    raise ValueError(f"Zones Silver input contains {null_zone_ids} null/blank zone_id rows")
if duplicate_zone_keys:
    raise ValueError(f"Zones Silver input contains {duplicate_zone_keys} duplicate zone_id keys")
if invalid_latitude_rows:
    raise ValueError(f"Zones Silver input contains {invalid_latitude_rows} invalid latitude rows")
if invalid_longitude_rows:
    raise ValueError(f"Zones Silver input contains {invalid_longitude_rows} invalid longitude rows")

prepared_duplicate_keys = (
    df_zones
    .groupBy("zone_id")
    .count()
    .filter(col("count") > 1)
    .count()
)
if prepared_duplicate_keys:
    raise ValueError(
        f"Zones Silver merge input contains {prepared_duplicate_keys} duplicate zone_id keys"
    )

prepared_rows = df_zones.count()
print(f"zones.quality.prepared_rows={prepared_rows}")
print(f"zones.quality.prepared_duplicate_zone_id_keys={prepared_duplicate_keys}")
display(df_zones)

In [ ]:
table_name = "ubereats_databricks1.silver.zones"

if not spark.catalog.tableExists(table_name):
    (
        df_zones.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )
else:
    target = DeltaTable.forName(spark, table_name)
    (
        target.alias("t")
        .merge(
            df_zones.alias("s"),
            "t.zone_id = s.zone_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )